# 03 - Training the Models

**Goal:** train the two supervised models on the prepared data from notebook 02.

- **Logistic Regression** - the simple, interpretable *baseline*.
- **Random Forest** - the stronger *primary* model (the one we later explain with SHAP).

We do not judge them here - that is notebook 04. Here we just train them and save them.

## Setup and load the prepared data

We load the file notebook 02 saved. It already contains the train/test split with the numbers cleaned, scaled and encoded - so there is nothing left to prepare.

In [1]:
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "outputs", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "outputs", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

data = joblib.load(os.path.join(PROCESSED_DIR, "train_test_data.joblib"))
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
feature_names = data["feature_names"]

print("Training rows:", X_train.shape[0], "| features:", X_train.shape[1])
print("Malicious in training set: {:.1%}".format(y_train.mean()))

Training rows: 597329 | features: 22
Malicious in training set: 85.4%


## A note on imbalance: `class_weight='balanced'`

Our attacks are the rare class. Left alone, a model can get a high score by mostly predicting 'benign'. `class_weight='balanced'` tells the model to **treat mistakes on the rare (attack) class as more costly**, so it pays proper attention to catching attacks. It is the simplest imbalance fix - no resampling, no extra data needed. We use it for both models.

## Model 1 - Logistic Regression (baseline)

A linear model: it learns a weight for each feature and adds them up to decide malicious vs benign. Simple and fully interpretable - good as a point of comparison.

`max_iter=1000` just gives it enough steps to settle (our data is scaled, so it converges easily).

In [2]:
logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train, y_train)
print("Logistic Regression trained.")
print("Training accuracy (not our final metric):", round(logreg.score(X_train, y_train), 3))

Logistic Regression trained.
Training accuracy (not our final metric): 0.816


## Model 2 - Random Forest (primary)

A Random Forest builds many decision trees on random parts of the data and lets them vote. It captures non-linear patterns a linear model can't, and it is the model we will explain with SHAP later.

- `n_estimators=100` - number of trees (more = a bit better but slower).
- `n_jobs=-1` - use all CPU cores (faster).
- `random_state=42` - reproducible results.

In [3]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print("Random Forest trained.")
print("Training accuracy (not our final metric):", round(rf.score(X_train, y_train), 3))

Random Forest trained.
Training accuracy (not our final metric): 0.955


## Save the trained models

We save both models so notebook 04 can load and evaluate them without retraining.

In [4]:
joblib.dump(logreg, os.path.join(MODELS_DIR, "logreg.joblib"))
joblib.dump(rf, os.path.join(MODELS_DIR, "random_forest.joblib"))
print("Saved models to:", MODELS_DIR)
print("Next: notebook 04 evaluates them with recall, precision, F1 and AUPRC.")

Saved models to: C:\Users\Asus\Desktop\Desktop\MSc Cybersecurity -NTU\Major Project\iot_anomaly_detection\outputs\models
Next: notebook 04 evaluates them with recall, precision, F1 and AUPRC.
